# Notebook 03 — Train Track A (Short Clinical CoT)

Goal: fine-tune Qwen2.5-1.5B with QLoRA on the same OpenMed rows as Track B,
but with **short clinical rationale + final answer** as the assistant output.
This is the primary system; Track B (Notebook 02) is the ablation baseline.

The only experimental variable between Track A and Track B is the assistant
output formatter. Everything else (model, data split, hyperparameters, seeds)
is identical — controlled by `configs/experiment_config.yaml`.

## Required Kaggle environment
- Accelerator: **GPU T4 x1**
- Internet: **On**
- Kaggle Secrets: `HF_TOKEN` (write scope), `WANDB_API_KEY`, `GROQ_API_KEY`

## 1. Bootstrap (clone, install, secrets)


In [ ]:
# Suppress noisy deprecation warnings from transformers/trl/unsloth.
import warnings
warnings.filterwarnings("ignore", message=".*AttentionMaskConverter.*")
warnings.filterwarnings("ignore", message=".*use_return_dict.*")
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")
warnings.filterwarnings("ignore", message=".*has new PAD/BOS/EOS tokens.*")
warnings.filterwarnings("ignore", message=".*Will smartly offload gradients.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")

import subprocess, sys, os

REPO_URL = "https://github.com/abhishek1998s/medical-reasoning-llm.git"
REPO_DIR = "/kaggle/working/medical-reasoning-llm"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
print("Contents:", os.listdir(REPO_DIR))

In [ ]:
# Clear any stale Unsloth compiled cache from a previous Kaggle session.
# Kaggle persists /kaggle/working/ across session stops. If your earlier
# session compiled Unsloth's wrappers against one PEFT version and a later
# session has a slightly drifted PEFT, the cached compile is invalid and
# you get cryptic errors like:
#   TypeError: dispatch_bnb_8bit() missing 1 required positional argument
# Always clear the cache before installing.
!rm -rf /kaggle/working/unsloth_compiled_cache

# Install the EXACT set of versions that worked end-to-end on Day 2's first
# successful run (109-min training). Pinning everything removes pip's resolver
# guesswork and keeps Day 3 (Track A) reproducible against the same combo.
!pip install -q --upgrade \
    unsloth==2026.4.8 \
    transformers==5.5.0 \
    trl==0.24.0 \
    peft==0.19.1 \
    bitsandbytes==0.49.2 \
    accelerate==1.13.0 \
    datasets==4.3.0 \
    wandb==0.19.4 \
    pyyaml

In [ ]:
# Verify the GPU stack imports cleanly.
import importlib

versions = {}
for pkg in ["unsloth", "transformers", "trl", "peft", "datasets",
            "bitsandbytes", "accelerate", "wandb"]:
    try:
        m = importlib.import_module(pkg)
        versions[pkg] = getattr(m, "__version__", "?")
    except Exception as e:
        versions[pkg] = f"FAILED: {e.__class__.__name__}: {e}"

for k, v in versions.items():
    print(f"  {k:20s} {v}")

failed = [k for k, v in versions.items() if str(v).startswith("FAILED")]
if failed:
    raise RuntimeError(f"Failed imports: {failed}. Restart kernel and re-run.")

In [ ]:
# Load Kaggle Secrets and log into HF.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login as hf_login

secrets = UserSecretsClient()

def _try_get(name):
    try:
        return secrets.get_secret(name)
    except Exception as e:
        print(f"  [skip] {name}: {e.__class__.__name__}")
        return None

os.environ["HF_TOKEN"]      = _try_get("HF_TOKEN")      or ""
os.environ["WANDB_API_KEY"] = _try_get("WANDB_API_KEY") or ""
os.environ["GROQ_API_KEY"]  = _try_get("GROQ_API_KEY")  or ""

print()
print("HF_TOKEN set:      ", bool(os.environ["HF_TOKEN"]))
print("WANDB_API_KEY set: ", bool(os.environ["WANDB_API_KEY"]))
print("GROQ_API_KEY set:  ", bool(os.environ["GROQ_API_KEY"]))

if os.environ["HF_TOKEN"]:
    hf_login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF login OK")

## 2. Train Track A (config-driven)


In [ ]:
# Build the training command from configs/experiment_config.yaml.
# Switching between DRY-RUN and FULL mode only requires editing the YAML —
# no changes needed here.
import yaml

cfg = yaml.safe_load(open(f"{REPO_DIR}/configs/experiment_config.yaml"))

hub_repo = f"{cfg['hub']['username']}/{cfg['hub']['repos']['trackA']}"
run_name = cfg['logging']['wandb_runs']['trackA']

print("Config summary:")
print(f"  num_train:       {cfg['dataset']['num_train']}")
print(f"  max_seq_length:  {cfg['model']['max_seq_length']}")
print(f"  batch_size:      {cfg['training']['per_device_train_batch_size']}")
print(f"  grad_accum:      {cfg['training']['gradient_accumulation_steps']}")
print(f"  hub_repo:        {hub_repo}")
print(f"  run_name:        {run_name}")
print()

cmd = (
    f"python train_sft.py"
    f" --track A_short"
    f" --num_samples {cfg['dataset']['num_train']}"
    f" --max_seq_length {cfg['model']['max_seq_length']}"
    f" --batch_size {cfg['training']['per_device_train_batch_size']}"
    f" --grad_accum {cfg['training']['gradient_accumulation_steps']}"
    f" --epochs {cfg['training']['epochs']}"
    f" --output_dir outputs/trackA"
    f" --run_name {run_name}"
    f" --push_to_hub"
    f" --hub_repo {hub_repo}"
)
print("Running:", cmd)
ret = os.system(cmd)
if ret != 0:
    raise RuntimeError(f"train_sft.py exited with code {ret}")

## 3. Smoke Check


In [ ]:
from pathlib import Path
import json

adapter = Path("outputs/trackA/final_adapter")
assert adapter.exists(), "Track A adapter was not saved — check training output above"

meta_path = adapter / "training_meta.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print(json.dumps(meta, indent=2)[:2000])
    assert meta.get("track") == "A_short", f"Unexpected track in meta: {meta.get('track')}"
    if meta.get("train_loss") is not None:
        assert meta["train_loss"] < 5.0, f"train_loss suspiciously high: {meta['train_loss']}"
        print(f"\ntrain_loss = {meta['train_loss']:.4f}  ✓")

print("\nTrack A adapter saved:", adapter)